In [ ]:
import time
import random
from collections import deque

def in_mt(mt):
    res = ""
    for i in range(3):
        row = ""
        for j in range(3):
            val = mt[i*3 + j]
            if val == 0:
                row += " [ ] "
            else:
                row += f"  {val}  "
        res += row + "\n"
    res += "-" * 20 + "\n"
    return res

def transition_state(state, action):
    pos = state.index(0)
    r, c = pos // 3, pos % 3
    target_pos = pos
    if action == "Lên" and r > 0:
        target_pos = pos - 3
    elif action == "Xuống" and r < 2:
        target_pos = pos + 3
    elif action == "Trái" and c > 0:
        target_pos = pos - 1
    elif action == "Phải" and c < 2:
        target_pos = pos + 1
        
    if target_pos == pos:
        return state
        
    new_state = list(state)
    new_state[pos], new_state[target_pos] = new_state[target_pos], new_state[pos]
    return tuple(new_state)

def transition_belief_state(belief_state, action):
    return frozenset(transition_state(s, action) for s in belief_state)

def count_inversions(state):
    arr = [val for val in state if val != 0]
    inv = 0
    for i in range(len(arr)):
        for j in range(i + 1, len(arr)):
            if arr[i] > arr[j]:
                inv += 1
    return inv

def calculate_belief_states(start_states, goal_states, max_depth=6, max_states_limit=5000):
    """
    Thuật toán khám phá và tính toán số lượng Trạng thái niềm tin (Belief States) thực tế trong không gian tìm kiếm.
    - start_states: Danh sách các trạng thái bắt đầu [[...], [...]]
    - goal_states: Danh sách các trạng thái đích [[...], [...]]
    """
    initial_belief = frozenset(tuple(s) for s in start_states)
    goal_set = set(tuple(g) for g in goal_states)
    
    # Kiểm tra chẵn lẻ để cảnh báo sớm khả năng đạt đích
    goal_parities = {count_inversions(g) % 2 for g in goal_set}
    unsolvable_states = []
    for s in initial_belief:
        if (count_inversions(s) % 2) not in goal_parities:
            unsolvable_states.append(list(s))
            
    explored = {initial_belief}
    queue = deque([(initial_belief, 0)]) # (belief_state, depth)
    
    parent = {initial_belief: (None, None)} # child -> (parent, action)
    
    size_counts = {1: 0, 2: 0}
    depth_stats = {} # depth -> {1: count, 2: count}
    goal_reached_depth = None
    goal_path = None
    limit_exceeded = False
    
    actions = ["Lên", "Xuống", "Trái", "Phải"]
    
    while queue:
        curr, depth = queue.popleft()
        
        if depth > max_depth:
            continue
            
        sz = len(curr)
        if sz in size_counts:
            size_counts[sz] += 1
            
        if depth not in depth_stats:
            depth_stats[depth] = {1: 0, 2: 0}
        depth_stats[depth][sz] += 1
        
        # Kiểm tra xem đã đạt Đích chưa (tập niềm tin là tập con không rỗng của tập đích)
        if len(curr) > 0 and curr.issubset(goal_set):
            if goal_reached_depth is None or depth < goal_reached_depth:
                goal_reached_depth = depth
                # Reconstruct path of belief states
                path = []
                temp = curr
                while temp != initial_belief:
                    p, act = parent[temp]
                    path.append((act, temp))
                    temp = p
                path.reverse()
                goal_path = path
                
        if len(explored) >= max_states_limit:
            limit_exceeded = True
            break
            
        for action in actions:
            child = transition_belief_state(curr, action)
            if child not in explored:
                explored.add(child)
                parent[child] = (curr, action)
                queue.append((child, depth + 1))
                
    return {
        "total_explored": len(explored),
        "size_counts": size_counts,
        "depth_stats": depth_stats,
        "goal_reached_depth": goal_reached_depth,
        "goal_path": goal_path,
        "limit_exceeded": limit_exceeded,
        "unsolvable_states": unsolvable_states,
        "goal_parities": list(goal_parities)
    }
